# Does chain-of-thought help? Measure it.

**Session 4 · small model (`llama3.2:3b`) vs big model (`gpt-oss:120b-cloud`)**

Compare CoT vs direct answering on a real reasoning set. Two findings to reach *from the
numbers*, not from folklore:

1. On the small model, CoT turns a near-total failure into near-perfect — a large, real gain.
2. On the big model, direct answering is already good, so CoT barely moves accuracy — it just
   costs tokens. **Whether a technique is worth it depends on the model.**

In [ ]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
import re
from utils import ask, count_tokens, SMALL_MODEL, BIG_MODEL
from eval import load_cases, run_eval, print_report, compare

cases = load_cases("../eval/datasets/math_reasoning.jsonl")  # 20 problems, each 2+ steps
print(f"{len(cases)} problems, e.g.: {cases[0]['input']}")

DIRECT = "{q}\nAnswer with a number only."
COT    = "{q}\nThink step by step, then end with a line 'FINAL: <number>'."

def last_number(s):
    nums = re.findall(r"-?\d+(?:\.\d+)?", s.replace(",", ""))
    return nums[-1] if nums else s.strip()

def num_match(output, expected):
    try:
        return abs(float(output) - float(expected)) < 1e-6
    except ValueError:
        return str(output).strip() == str(expected).strip()

def make(template, model, is_cot=False):
    def answer(q):
        out = ask(template.format(q=q), model=model)
        return last_number(out.split("FINAL:")[-1] if is_cot else out)
    return answer


20 problems, e.g.: A shirt costs $40 after a 20% discount. What was the original price in dollars?


### 1. Small model: does CoT help?

`compare()` runs both prompts 3 times over all 20 problems and checks whether the gap
between them is bigger than the run-to-run noise. If it is, the effect is real.

In [2]:
compare(
    cases,
    make(DIRECT, SMALL_MODEL),
    make(COT, SMALL_MODEL, is_cot=True),
    labels=("direct", "chain-of-thought"),
    scorer=num_match,
    repeats=3,
)


  direct                     20%   (spread 0% over 3 runs)
  chain-of-thought           90%   (spread 0% over 3 runs)
  gap +70%   vs   run-to-run noise 0%   ->   REAL


{'a': {'n': 20,
  'repeats': 3,
  'accuracies': [0.2, 0.2, 0.2],
  'acc_mean': 0.20000000000000004,
  'acc_min': 0.2,
  'acc_max': 0.2,
  'spread': 0.0,
  'accuracy': 0.20000000000000004,
  'passed': 4,
  'results': [{'input': 'A shirt costs $40 after a 20% discount. What was the original price in dollars?',
    'expected': '50',
    'output': '60',
    'pass': False},
   {'input': 'If 3 pens cost $2.10, what do 10 pens cost in dollars?',
    'expected': '7',
    'output': '6.30',
    'pass': False},
   {'input': 'Tom is twice as old as Sara. In 5 years their ages sum to 40. How old is Tom now?',
    'expected': '20',
    'output': '25',
    'pass': False},
   {'input': 'A bookstore had 240 books. It sold 35% on Monday and a quarter of the remainder on Tuesday. How many books are left?',
    'expected': '117',
    'output': '205',
    'pass': False},
   {'input': 'A tank holds 500 litres. It leaks 12 litres per hour for 6 hours, then 200 litres are added. How many litres are in the tan

### 2. What does CoT cost?

CoT's price is output tokens: the model writes a paragraph instead of a number. Measure it.

In [ ]:
def avg_output_tokens(template, model, is_cot=False):
    tot = 0
    for c in cases:
        out = ask(template.format(q=c["input"]), model=model)
        tot += count_tokens(out)
    return tot / len(cases)

d_tok = avg_output_tokens(DIRECT, SMALL_MODEL)
c_tok = avg_output_tokens(COT, SMALL_MODEL, is_cot=True)
print(f"direct: ~{d_tok:.0f} output tokens/answer")
print(f"CoT:    ~{c_tok:.0f} output tokens/answer   ({c_tok / max(d_tok, 1):.0f}x more)")


### 3. Big model: same comparison

Run the identical test on the frontier model. Expect: direct answering already strong, so
the CoT gap shrinks toward the noise floor and `compare()` returns **INCONCLUSIVE**. Same
technique, different verdict — because the model changed.

_(Uses the Ollama Cloud model. If it is unavailable the cell prints a note and skips.)_

In [ ]:
try:
    compare(
        cases,
        make(DIRECT, BIG_MODEL),
        make(COT, BIG_MODEL, is_cot=True),
        labels=("direct", "chain-of-thought"),
        scorer=num_match,
        repeats=2,
    )
except Exception as e:
    print(f"skipped big-model run: {type(e).__name__}: {e}")


## Your turn - vary the example

1. Add 3 harder problems where you expect direct answering to fail.
2. Try a middle option: "give a one-line reason, then the answer". Where does it land?
3. Keep the winner. Is the extra CoT token cost worth the accuracy gain here?
